In [1]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"] = "xx"

In [3]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 19 file(s) under /src/
src/cl_strategies/ewc.py -> /content/src/cl_strategies/ewc.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/kd/mseLoss.py -> /content/src/kd/mseLoss.py
src/meta_learning/buffers.py -> /content/src/meta_learning/buffers.py
src/meta_learning/data.py -> /content/src/meta_learning/data.py
src/meta_learning/embeddings.py -> /content/src/meta_learning/embeddings.py
src/meta_learning/models.py -> /content/src/meta_le

In [4]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh    # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
Dataset path: /kaggle/input/ucf101-action-recognition

Detecting path and updating config.py...
Found in kagglehub cache: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
DATASET_ROOT = '/root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/'
OUTPUT_ROOT  = '/content/ucf101-processed/'

Setup complete!
config.py updated on Drive.
Restarting kernel in 5 seconds...
Dataset cache stays on disk.
Run Cell 3 after restart to verify.
Restarting in 5...
Restarting in 4...
Restarting in 3...
Restarting in 2...
Restarting in 1...


In [5]:
# imports
%run /content/src/imports.py

Using device: cuda


## **Paths & Hyperparameters**

In [6]:
# preprocessing roots
TASK0_ROOT           = "./UCF101/processed_task0"
EXEMPLAR_ROOT        = "./UCF101/processed_exemplars"
SUBSET1_ROOT         = "./UCF101/processed_subset1"
LIMITED_SUBSET1_ROOT = "./UCF101/processed_limited_subset1"
SUBSET2_ROOT         = "./UCF101/processed_subset2"
LIMITED_SUBSET2_ROOT = "./UCF101/processed_limited_subset2"

# embedding roots
TASK0_EMB_ROOT    = "./UCF101/emb_task0"
EXEMPLAR_EMB_ROOT = "./UCF101/emb_exemplars"
SUBSET1_EMB_ROOT  = "./UCF101/emb_subset1"
LIM_SUB1_EMB_ROOT = "./UCF101/emb_limited_subset1"
SUBSET2_EMB_ROOT  = "./UCF101/emb_subset2"
LIM_SUB2_EMB_ROOT = "./UCF101/emb_limited_subset2"

ALL_EMB_ROOTS = [
    TASK0_EMB_ROOT, EXEMPLAR_EMB_ROOT,
    SUBSET1_EMB_ROOT, LIM_SUB1_EMB_ROOT,
    SUBSET2_EMB_ROOT, LIM_SUB2_EMB_ROOT,
]


In [7]:
LIMIT = 3

NEW_REPEAT        = 4
LABEL_SMOOTHING   = 0.1

REH_LR_LSTM       = 1e-4
REH_LR_FC         = 3e-3
REPLAY_RATIO      = 0.6

LAMBDA_DISTILL_T1 = 0.4
LAMBDA_DISTILL_T2 = 0.7

T1_EPOCHS         = 12
T2_EPOCHS         = 14

REP_INNER_LR      = 0.03
REP_INNER_STEPS   = 10
REP_EPSILON       = 0.18

REP_K_SUPPORT     = 3
REP_K_QUERY       = 3

LAMBDA_QRY        = 1.2
LAMBDA_NEW_REP    = 1.5

LAMBDA_KD_REPTILE = 0.5
LAMBDA_REPLAY     = 0.3  

LAMBDA_KD_T1      = 0.4
LAMBDA_KD_T2      = 0.8

REH_LR              = 1e-4

# Class indexing
ALL_CLASSES_LIST    = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx        = {cls: i for i, cls in enumerate(ALL_CLASSES_LIST)}
N0                  = len(cfg.SELECTED_CLASSES)
N1                  = len(cfg.TASK_1)
N2                  = len(cfg.TASK_2)
NUM_CLASSES_T1 = N0 + N1
NUM_CLASSES_T2 = N0 + N1 + N2

## **Preprocess**

In [8]:
# preprocess just the val,test on the old classes
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, splits=["test"])          

# old classes trian limited for reply buffer
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=EXEMPLAR_ROOT, max_samples=LIMIT, splits=["train"])

# task 1 few-shot
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=LIMITED_SUBSET1_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, splits=["test"])

# task 2 few-shot
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=LIMITED_SUBSET2_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, splits=["test"])


--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_exemplars
Classes: 10 | Limit: 3

Processing split: train



Done! Dataset ready at: ./UCF101/processed_exemplars

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_limited_subset1
Classes: 3 | Limit: 3

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_limited_subset2
Classes: 3 | Limit: 3

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset2

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset2

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset2


## **Upload Model**

In [9]:
model = ResNet50LSTM(num_classes=N0).to(device)
model.load_state_dict(torch.load(cfg.RESNET50_PATH, map_location=device))

<All keys matched successfully>

In [10]:
model_embeddings = ResNet50Trunk(model).to(device)
model_embeddings.eval() # evaluation
del model  # keep only the backbone

## **Exctract embeddings**

In [11]:
extraction_plan = [
    (TASK0_ROOT,           TASK0_EMB_ROOT,    "Task 0        -> val&test"),
    (EXEMPLAR_ROOT,        EXEMPLAR_EMB_ROOT, f"Exemplars     -> {LIMIT}/cls train  <- replay buffer"),
    (LIMITED_SUBSET1_ROOT, LIM_SUB1_EMB_ROOT, f"Task 1 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET1_ROOT,         SUBSET1_EMB_ROOT,  "Task 1        -> val & test"),
    (LIMITED_SUBSET2_ROOT, LIM_SUB2_EMB_ROOT, f"Task 2 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET2_ROOT,         SUBSET2_EMB_ROOT,  "Task 2        -> val & test"),
]

# delete existing dirs
for d in ALL_EMB_ROOTS:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"  Deleted stale dir: {d}")

for clip_root, emb_root, label in extraction_plan:
    print(f"  -> {label}")
    extract_embeddings(clip_root, emb_root, model_embeddings, device)
    torch.cuda.empty_cache()

  Deleted stale dir: ./UCF101/emb_task0
  Deleted stale dir: ./UCF101/emb_exemplars
  Deleted stale dir: ./UCF101/emb_subset1
  Deleted stale dir: ./UCF101/emb_limited_subset1
  Deleted stale dir: ./UCF101/emb_subset2
  Deleted stale dir: ./UCF101/emb_limited_subset2
  -> Task 0        -> val&test


  -> Exemplars     -> 3/cls train  <- replay buffer


  -> Task 1 train  -> 3/cls (few-shot)


  -> Task 1        -> val & test


  -> Task 2 train  -> 3/cls (few-shot)


  -> Task 2        -> val & test


## **Loaders**

In [12]:
# DATASETS
# Task 0: val & test
task0_val_ds  = UCF101Embeddings(f"{TASK0_EMB_ROOT}/val",  class_to_idx)
task0_test_ds = UCF101Embeddings(f"{TASK0_EMB_ROOT}/test", class_to_idx)

# exemplars limit classes from T0 train
exemplar_train_ds = UCF101Embeddings(f"{EXEMPLAR_EMB_ROOT}/train", class_to_idx)

# Task 1: few-shot
task1_train_ds = UCF101Embeddings(f"{LIM_SUB1_EMB_ROOT}/train", class_to_idx)
task1_val_ds   = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/val",    class_to_idx)
task1_test_ds  = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/test",   class_to_idx)

# Task 2: few-shot
task2_train_ds = UCF101Embeddings(f"{LIM_SUB2_EMB_ROOT}/train", class_to_idx)
task2_val_ds   = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/val",    class_to_idx)
task2_test_ds  = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/test",   class_to_idx)


# DATALOADERS
task0_val_loader      = _dl(task0_val_ds)
task0_test_loader     = _dl(task0_test_ds)
exemplar_train_loader = _dl(exemplar_train_ds)   # shuffle=False

task1_train_loader = _dl(task1_train_ds, shuffle=True)
task1_val_loader   = _dl(task1_val_ds)
task1_test_loader  = _dl(task1_test_ds)

task2_train_loader = _dl(task2_train_ds, shuffle=True)
task2_val_loader   = _dl(task2_val_ds)
task2_test_loader  = _dl(task2_test_ds)

# combined loaders
combined_val_t1_loader = _dl(ConcatDataset([task0_val_ds, task1_val_ds]), shuffle=True)
combined_val_t2_loader = _dl(ConcatDataset([task0_val_ds, task1_val_ds, task2_val_ds]), shuffle=True)

combined_test_t1_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds]))
combined_test_t2_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds, task2_test_ds]))

combined_exemplar_t2_ds = ConcatDataset([exemplar_train_ds, task1_train_ds])
combined_exemplar_t2_loader = _dl(combined_exemplar_t2_ds)

## **Buffers**

In [13]:
# class names in index order
num_classes = len(class_to_idx)
class_names = [None] * num_classes
for cls_name, idx in class_to_idx.items():
    class_names[idx] = cls_name


In [14]:
(replay_t1, episode_t1, n_way_eff_t1) = build_task1_buffers(exemplar_train_loader, limit=8, rep_n_way=3)
(replay_t2, episode_t2, n_way_eff_t2) = build_task2_buffers(exemplar_train_loader, task1_train_loader, limit=8, rep_n_way=3)

[T1 buffers] ReplayBuffer(total=80, classes={0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 8})
[T1 episode] EpisodeBuffer(total=80, new_classes=[], per_class={0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 8})
[T1] Effective n_way = 3
[T2 buffers] ReplayBuffer(total=104, classes={0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 8, 12: 8, 10: 8, 11: 8})
[T2 episode] EpisodeBuffer(total=104, new_classes=[10, 11, 12], per_class={0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 8, 11: 8, 10: 8, 12: 8})
[T2] Effective n_way = 3


In [15]:
buffer_comparison = {
    "Task 1 Replay": replay_t1,
    "Task 1 Episode": episode_t1,
    "Task 2 Replay": replay_t2,
    "Task 2 Episode": episode_t2
}

fig = plot_buffer_composition(buffer_comparison, class_names=class_names)

fig.show()

## **Evaluate & Record ResNet50LSTM**

In [17]:
model_embeddings = fresh_model(num_classes=NUM_CLASSES_T1, cfg=cfg, device=device, lstm_hidden=256)
model_embeddings.eval()

W0_base_t1 = {k: v.cpu().clone() for k, v in model_embeddings.state_dict().items()}

record(
    name="Base ResNet50LSTM",
    model=model_embeddings,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(NUM_CLASSES_T1)],
    W0=W0_base_t1,
)

model_embeddings = fresh_model(num_classes=NUM_CLASSES_T2, cfg=cfg, device=device, lstm_hidden=256)
model_embeddings.eval()

W0_base_t2 = {k: v.cpu().clone() for k, v in model_embeddings.state_dict().items()}

record(
    name="Base ResNet50LSTM",
    model=model_embeddings,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(NUM_CLASSES_T2)],
    W0=W0_base_t2,
)

# cleanup
del model_embeddings
del W0_base_t1
del W0_base_t2
torch.cuda.empty_cache()

  [Base ResNet50LSTM] Old: 0.1111 | New: 0.1132 | All: 0.1116
  [Base ResNet50LSTM] Old: 0.0268 | New: 0.0364 | All: 0.0287


In [18]:
plot_confusion({k: results_t2[k] for k in results_t2.keys() if "Base ResNet50LSTM" in k}, task_label="T2", global_class_names=class_names).show()

In [19]:
teacher = make_frozen_teacher(
    num_classes   = N0,  
    cfg           = cfg,
    device        = device,
    lstm_hidden   = 256,
    checkpoint_path = cfg.RESNET50_PATH,
)

def make_student_t1():
    return build_student(
        num_classes    = NUM_CLASSES_T1,
        cfg            = cfg,
        device         = device,
        lstm_hidden    = 256,
        num_old_classes= N0,
    )

def make_student_t2(trained_t1_model):
    t2 = fresh_model(
        num_classes = NUM_CLASSES_T2,
        cfg         = cfg,
        device      = device,
        lstm_hidden = 256,
    )
    sd_t1 = trained_t1_model.state_dict()
    sd_t2 = t2.state_dict()
    for k in sd_t2:
        if k.startswith("lstm"):
            sd_t2[k] = sd_t1[k].clone()
    with torch.no_grad():
        rows = NUM_CLASSES_T1
        sd_t2["fc.weight"][:rows] = sd_t1["fc.weight"][:rows].clone()
        sd_t2["fc.bias"][:rows]   = sd_t1["fc.bias"][:rows].clone()
    t2.load_state_dict(sd_t2)
    return t2.to(device)

## **Task 1**

- Rehersial no-kd

In [26]:
m_reh_nokd_t1 = make_student_t1()
m_reh_nokd_t1 = train_rehearsal(
    model=m_reh_nokd_t1, teacher=None,
    train_loader=task1_train_loader, val_loader=task1_val_loader,
    replay_buffer=replay_t1, optimizer=make_rehearsal_optimizer(m_reh_nokd_t1, lr=REH_LR),
    device=device, num_old_classes=N0,
    num_classes=NUM_CLASSES_T1,
    epochs=T1_EPOCHS, new_repeat=NEW_REPEAT,
    replay_ratio=REPLAY_RATIO, lambda_distill=0.0,
    label_smoothing=LABEL_SMOOTHING, kd=False, patience=3,
)
record("Rehearsal No KD T1", m_reh_nokd_t1, results_t1,
       task0_test_loader, task1_test_loader, combined_test_t1_loader,
       device=device,
       embed_loader=combined_test_t1_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T1)])

[Rehearsal] Epoch 1/12 | Loss: 20.2353 | ValAcc: 0.3800
[Rehearsal] Epoch 2/12 | Loss: 11.9547 | ValAcc: 0.6200
[Rehearsal] Epoch 3/12 | Loss: 8.8380 | ValAcc: 0.6200
[Rehearsal] Epoch 4/12 | Loss: 7.5547 | ValAcc: 0.5800
[Rehearsal] Epoch 5/12 | Loss: 7.0344 | ValAcc: 0.6000
[Rehearsal] Early stop at epoch 5, best ValAcc: 0.6200
  [Rehearsal No KD T1] Old: 0.9474 | New: 0.7547 | All: 0.9018


In [27]:
plot_confusion({k: results_t1[k] for k in results_t1.keys() if "Rehearsal No KD T1" in k}, task_label="T1", global_class_names=class_names).show()

- Rehersial kd

In [29]:
# Rehearsal KD T1
m_reh_kd_t1 = make_student_t1()
m_reh_kd_t1 = train_rehearsal(
    model=m_reh_kd_t1, teacher=teacher,
    train_loader=task1_train_loader, val_loader=task1_val_loader,
    replay_buffer=replay_t1, optimizer=make_rehearsal_optimizer(m_reh_kd_t1, lr=REH_LR),
    device=device, num_old_classes=N0,
    num_classes=NUM_CLASSES_T1,
    epochs=T1_EPOCHS, new_repeat=NEW_REPEAT,
    replay_ratio=REPLAY_RATIO, lambda_distill=LAMBDA_DISTILL_T1,
    label_smoothing=LABEL_SMOOTHING, kd=True, patience=3,
)
record("Rehearsal KD T1", m_reh_kd_t1, results_t1,
       task0_test_loader, task1_test_loader, combined_test_t1_loader,
       device=device,
       embed_loader=combined_test_t1_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T1)])

[Rehearsal] Epoch 1/12 | Loss: 21.7364 | ValAcc: 0.3400
[Rehearsal] Epoch 2/12 | Loss: 13.6963 | ValAcc: 0.5200
[Rehearsal] Epoch 3/12 | Loss: 10.3020 | ValAcc: 0.6200
[Rehearsal] Epoch 4/12 | Loss: 8.6077 | ValAcc: 0.6000
[Rehearsal] Epoch 5/12 | Loss: 8.0194 | ValAcc: 0.5600
[Rehearsal] Epoch 6/12 | Loss: 7.7303 | ValAcc: 0.5800
[Rehearsal] Early stop at epoch 6, best ValAcc: 0.6200
  [Rehearsal KD T1] Old: 0.9474 | New: 0.7736 | All: 0.9062


In [30]:
plot_confusion({k: results_t1[k] for k in results_t1.keys() if "Rehearsal KD T1" in k}, task_label="T1", global_class_names=class_names).show()

- Reptile no-kd

In [31]:
m_rep_nokd_t1 = make_student_t1()
m_rep_nokd_t1 = train_reptile(
    model=m_rep_nokd_t1, episode_buffer=episode_t1,
    train_loader=task1_train_loader, val_loader=task1_val_loader,
    device=device, teacher=None, num_old_classes=N0,
    num_classes=NUM_CLASSES_T1, n_way=10,
    k_support=REP_K_SUPPORT, k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR, inner_steps=REP_INNER_STEPS, epsilon=REP_EPSILON,
    epochs=T1_EPOCHS, new_repeat=NEW_REPEAT,
    lambda_kd=0.0, lambda_qry=LAMBDA_QRY, lambda_new=LAMBDA_NEW_REP,
    label_smoothing=LABEL_SMOOTHING, kd=False, patience=3,
)
record("Reptile No KD T1", m_rep_nokd_t1, results_t1,
       task0_test_loader, task1_test_loader, combined_test_t1_loader,
       device=device,
       embed_loader=combined_test_t1_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T1)])

[Reptile] Epoch 1/12 | ValAcc: 0.5200
[Reptile] Epoch 2/12 | ValAcc: 0.6400
[Reptile] Epoch 3/12 | ValAcc: 0.5600
[Reptile] Epoch 4/12 | ValAcc: 0.5400
[Reptile] Epoch 5/12 | ValAcc: 0.5600
[Reptile] Early stop at epoch 5, best ValAcc: 0.6400
  [Reptile No KD T1] Old: 0.9532 | New: 0.7736 | All: 0.9107


In [32]:
plot_confusion({k: results_t1[k] for k in results_t1.keys() if "Reptile No KD T1" in k}, task_label="T1", global_class_names=class_names).show()

- Reptile Kd

In [36]:
m_rep_kd_t1 = make_student_t1()
m_rep_kd_t1 = train_reptile(
    model=m_rep_kd_t1, episode_buffer=episode_t1,
    train_loader=task1_train_loader, val_loader=task1_val_loader,
    device=device, teacher=teacher, num_old_classes=N0,
    num_classes=NUM_CLASSES_T1, n_way=10,
    k_support=REP_K_SUPPORT, k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR, inner_steps=REP_INNER_STEPS, epsilon=REP_EPSILON,
    epochs=T1_EPOCHS, new_repeat=NEW_REPEAT,
    lambda_kd=LAMBDA_KD_REPTILE, lambda_qry=LAMBDA_QRY, lambda_new=LAMBDA_NEW_REP,
    label_smoothing=LABEL_SMOOTHING, kd=True, patience=3,
)
record("Reptile KD T1", m_rep_kd_t1, results_t1,
       task0_test_loader, task1_test_loader, combined_test_t1_loader,
       device=device,
       embed_loader=combined_test_t1_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T1)])

[Reptile] Epoch 1/12 | ValAcc: 0.5600
[Reptile] Epoch 2/12 | ValAcc: 0.6600
[Reptile] Epoch 3/12 | ValAcc: 0.5600
[Reptile] Epoch 4/12 | ValAcc: 0.5800
[Reptile] Epoch 5/12 | ValAcc: 0.5600
[Reptile] Early stop at epoch 5, best ValAcc: 0.6600
  [Reptile KD T1] Old: 0.9474 | New: 0.8491 | All: 0.9241


In [37]:
plot_confusion({k: results_t1[k] for k in results_t1.keys() if "Reptile KD T1" in k}, task_label="T1", global_class_names=class_names).show()

## **TASK 2**

- rehersial no-kd

In [39]:
teacher_reh_t1 = copy.deepcopy(m_reh_kd_t1).eval()
for p in teacher_reh_t1.parameters(): p.requires_grad = False

teacher_rep_t1 = copy.deepcopy(m_rep_kd_t1).eval()
for p in teacher_rep_t1.parameters(): p.requires_grad = False

In [40]:
m_reh_nokd_t2 = make_student_t2(m_reh_nokd_t1)
m_reh_nokd_t2 = train_rehearsal(
    model=m_reh_nokd_t2, teacher=None,
    train_loader=task2_train_loader, val_loader=task2_val_loader,
    replay_buffer=replay_t2, optimizer=make_rehearsal_optimizer(m_reh_nokd_t2, lr=REH_LR),
    device=device, num_old_classes=N0 + N1,
    num_classes=NUM_CLASSES_T2,
    epochs=T2_EPOCHS, new_repeat=NEW_REPEAT,
    replay_ratio=REPLAY_RATIO, lambda_distill=0.0,
    label_smoothing=LABEL_SMOOTHING, kd=False, patience=3,
)
record("Rehearsal No KD T2", m_reh_nokd_t2, results_t2,
       combined_test_t1_loader, task2_test_loader, combined_test_t2_loader,
       device=device,
       embed_loader=combined_test_t2_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T2)])

[Rehearsal] Epoch 1/14 | Loss: 23.6374 | ValAcc: 0.1887
[Rehearsal] Epoch 2/14 | Loss: 13.8254 | ValAcc: 0.8113
[Rehearsal] Epoch 3/14 | Loss: 9.5393 | ValAcc: 0.8491
[Rehearsal] Epoch 4/14 | Loss: 7.9687 | ValAcc: 0.8113
[Rehearsal] Epoch 5/14 | Loss: 7.3691 | ValAcc: 0.8302
[Rehearsal] Epoch 6/14 | Loss: 7.2234 | ValAcc: 0.8302
[Rehearsal] Early stop at epoch 6, best ValAcc: 0.8491
  [Rehearsal No KD T2] Old: 0.7277 | New: 0.7818 | All: 0.7384


In [47]:
plot_confusion({k: results_t2[k] for k in results_t2.keys() if "Rehearsal No KD T2" in k}, task_label="T2", global_class_names=class_names).show()

- rehersial kd

In [41]:
m_reh_kd_t2 = make_student_t2(m_reh_kd_t1)
m_reh_kd_t2 = train_rehearsal(
    model=m_reh_kd_t2, teacher=teacher_reh_t1,
    train_loader=task2_train_loader, val_loader=task2_val_loader,
    replay_buffer=replay_t2, optimizer=make_rehearsal_optimizer(m_reh_kd_t2, lr=REH_LR),
    device=device, num_old_classes=N0 + N1,
    num_classes=NUM_CLASSES_T2,
    epochs=T2_EPOCHS, new_repeat=NEW_REPEAT,
    replay_ratio=REPLAY_RATIO, lambda_distill=LAMBDA_DISTILL_T2,
    label_smoothing=LABEL_SMOOTHING, kd=True, patience=3,
)
record("Rehearsal KD T2", m_reh_kd_t2, results_t2,
       combined_test_t1_loader, task2_test_loader, combined_test_t2_loader,
       device=device,
       embed_loader=combined_test_t2_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T2)])

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning:

RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)



[Rehearsal] Epoch 1/14 | Loss: 24.4501 | ValAcc: 0.0566
[Rehearsal] Epoch 2/14 | Loss: 15.8701 | ValAcc: 0.5849
[Rehearsal] Epoch 3/14 | Loss: 11.9539 | ValAcc: 0.7925
[Rehearsal] Epoch 4/14 | Loss: 9.9966 | ValAcc: 0.8113
[Rehearsal] Epoch 5/14 | Loss: 9.1723 | ValAcc: 0.8113
[Rehearsal] Epoch 6/14 | Loss: 8.6953 | ValAcc: 0.8113
[Rehearsal] Epoch 7/14 | Loss: 8.5114 | ValAcc: 0.8113
[Rehearsal] Early stop at epoch 7, best ValAcc: 0.8113
  [Rehearsal KD T2] Old: 0.7366 | New: 0.7273 | All: 0.7348


In [48]:
plot_confusion({k: results_t2[k] for k in results_t2.keys() if "Rehearsal KD T2" in k}, task_label="T2", global_class_names=class_names).show()

- Reptile no-kd

In [42]:
m_rep_nokd_t2 = make_student_t2(m_rep_nokd_t1)
m_rep_nokd_t2 = train_reptile(
    model=m_rep_nokd_t2, episode_buffer=episode_t2,
    train_loader=task2_train_loader, val_loader=task2_val_loader,
    device=device, teacher=None, num_old_classes=N0 + N1,
    num_classes=NUM_CLASSES_T2, n_way=10,
    k_support=REP_K_SUPPORT, k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR, inner_steps=REP_INNER_STEPS, epsilon=REP_EPSILON,
    epochs=T2_EPOCHS, new_repeat=NEW_REPEAT,
    lambda_kd=0.0, lambda_qry=LAMBDA_QRY, lambda_new=LAMBDA_NEW_REP,
    label_smoothing=LABEL_SMOOTHING, kd=False, patience=3,
)
record("Reptile No KD T2", m_rep_nokd_t2, results_t2,
       combined_test_t1_loader, task2_test_loader, combined_test_t2_loader,
       device=device,
       embed_loader=combined_test_t2_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T2)])

[Reptile] Epoch 1/14 | ValAcc: 0.6038
[Reptile] Epoch 2/14 | ValAcc: 0.8302
[Reptile] Epoch 3/14 | ValAcc: 0.7358
[Reptile] Epoch 4/14 | ValAcc: 0.6981
[Reptile] Epoch 5/14 | ValAcc: 0.6981
[Reptile] Early stop at epoch 5, best ValAcc: 0.8302
  [Reptile No KD T2] Old: 0.8214 | New: 0.6364 | All: 0.7849


In [49]:
plot_confusion({k: results_t2[k] for k in results_t2.keys() if "Reptile No KD T2" in k}, task_label="T2", global_class_names=class_names).show()

- reptile kd

In [43]:
m_rep_kd_t2 = make_student_t2(m_rep_kd_t1)
m_rep_kd_t2 = train_reptile(
    model=m_rep_kd_t2, episode_buffer=episode_t2,
    train_loader=task2_train_loader, val_loader=task2_val_loader,
    device=device, teacher=teacher_rep_t1,
    num_old_classes=N0 + N1,
    num_classes=NUM_CLASSES_T2, n_way=10,
    k_support=REP_K_SUPPORT, k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR, inner_steps=REP_INNER_STEPS, epsilon=REP_EPSILON,
    epochs=T2_EPOCHS, new_repeat=NEW_REPEAT,
    lambda_kd=LAMBDA_KD_REPTILE, lambda_qry=LAMBDA_QRY, lambda_new=LAMBDA_NEW_REP,
    label_smoothing=LABEL_SMOOTHING, kd=True, patience=3,
)
record("Reptile KD T2", m_rep_kd_t2, results_t2,
       combined_test_t1_loader, task2_test_loader, combined_test_t2_loader,
       device=device,
       embed_loader=combined_test_t2_loader,
       class_names=[str(i) for i in range(NUM_CLASSES_T2)])

[Reptile] Epoch 1/14 | ValAcc: 0.5660
[Reptile] Epoch 2/14 | ValAcc: 0.7547
[Reptile] Epoch 3/14 | ValAcc: 0.7358
[Reptile] Epoch 4/14 | ValAcc: 0.6981
[Reptile] Epoch 5/14 | ValAcc: 0.6792
[Reptile] Early stop at epoch 5, best ValAcc: 0.7547
  [Reptile KD T2] Old: 0.8527 | New: 0.6909 | All: 0.8208


In [50]:
plot_confusion({k: results_t2[k] for k in results_t2.keys() if "Reptile KD T2" in k}, task_label="T2", global_class_names=class_names).show()

In [44]:
fig = plot_accuracy(results_t2, task_label="T2")
fig.show()